**Imports**

In [2]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

**Spark Session**

In [3]:
spark = SparkSession.builder \
    .appName("NYC_Taxi_Preprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

#Kritik hatalar
spark.sparkContext.setLogLevel("ERROR")

print("Spark Oturumu Başarıyla Başlatıldı!")
print(f"Spark Versiyonu: {spark.version}")

Spark Oturumu Başarıyla Başlatıldı!
Spark Versiyonu: 3.5.1


**Klasör Yolları**

In [4]:
RAW_DIR = "data/raw"
CLEAN_DIR = "data/clean/yellow_tripdata_2023"

Ay bazında okumak için döngü

In [5]:
for month in range(1, 13):
    month_str = f"{month:02d}"
    file_name = f"yellow_tripdata_2023-{month_str}.parquet"
    file_path = os.path.join(RAW_DIR, file_name)

    # O aya ait klasör yoksa hata vermeden atla
    if not os.path.exists(file_path):
        print(f"Atlanıyor: {file_path} bulunamadı.")
        continue

    print(f"[{month_str}/12] {file_name} işleniyor...")
    start_time = time.perf_counter()

    # Tek bir ay için parquet dosyasını oku
    df_month = spark.read.parquet(file_path)

    # Projection Prunning
    selected_columns = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'total_amount']
    df_filtered = df_month.select(*selected_columns)

    # Yeni sütunlar ekle
    df_filtered = df_filtered.withColumn("pickup_year", F.year(F.col("tpep_pickup_datetime")))\
                             .withColumn("pickup_month", F.month(F.col("tpep_pickup_datetime")))
    
    # Veriyi temizle
    df_cleaned = df_filtered.filter(
        (F.col("pickup_year") == 2023) &
        (F.col("pickup_month") == month) &
        (F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime")) &
        (F.col("total_amount") > 0.0) & (F.col("total_amount") < 500.0) &
        (F.col("trip_distance") > 0.0) & (F.col("trip_distance") < 100.0) &
        (F.col("PULocationID").between(1, 263)) & 
        (F.col("DOLocationID").between(1, 263))
    )

    # Seyahat süresini hesapla ve yeni bir sütun olarak ekle
    df_cleaned = df_cleaned.withColumn(
        "trip_duration_seconds",
        F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
    ).filter((F.col("trip_duration_seconds") > 30) & (F.col("trip_duration_seconds") < 18000))

    # Duplicate verileri yok et
    df_cleaned = df_cleaned.dropDuplicates()

    # Null değerleri yok et
    df_cleaned = df_cleaned.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime", 
                                           "PULocationID", "DOLocationID", "trip_distance", 
                                           "total_amount"])
    
    # Temizlenmiş veriyi diske append et ve partiton et
    df_cleaned.write \
        .mode("append") \
        .partitionBy("pickup_year", "pickup_month") \
        .parquet(CLEAN_DIR)
    
    end_time = time.perf_counter()
    print(f"-> {file_name} tamamlandı. Süre: {end_time - start_time:.2f} saniye.\n")

print("YELLOW TAXI VERİ SETİ BAŞARIYLA TEMİZLENDİ VE OPTİMİZE EDİLDİ")


[01/12] yellow_tripdata_2023-01.parquet işleniyor...
-> yellow_tripdata_2023-01.parquet tamamlandı. Süre: 11.00 saniye.

[02/12] yellow_tripdata_2023-02.parquet işleniyor...
-> yellow_tripdata_2023-02.parquet tamamlandı. Süre: 4.79 saniye.

[03/12] yellow_tripdata_2023-03.parquet işleniyor...
-> yellow_tripdata_2023-03.parquet tamamlandı. Süre: 4.75 saniye.

[04/12] yellow_tripdata_2023-04.parquet işleniyor...
-> yellow_tripdata_2023-04.parquet tamamlandı. Süre: 4.33 saniye.

[05/12] yellow_tripdata_2023-05.parquet işleniyor...
-> yellow_tripdata_2023-05.parquet tamamlandı. Süre: 4.47 saniye.

[06/12] yellow_tripdata_2023-06.parquet işleniyor...
-> yellow_tripdata_2023-06.parquet tamamlandı. Süre: 4.14 saniye.

[07/12] yellow_tripdata_2023-07.parquet işleniyor...
-> yellow_tripdata_2023-07.parquet tamamlandı. Süre: 3.89 saniye.

[08/12] yellow_tripdata_2023-08.parquet işleniyor...
-> yellow_tripdata_2023-08.parquet tamamlandı. Süre: 2.30 saniye.

[09/12] yellow_tripdata_2023-09.parquet

In [7]:
raw = spark.read.parquet("data/raw/yellow_tripdata_2023-*.parquet")
print("Ham:", raw.count())
print("Negatif ücret:", raw.filter("total_amount <= 0").count())
print("Yıl != 2023:", raw.filter("year(tpep_pickup_datetime) != 2023").count())
print("Geçersiz zone:", raw.filter("PULocationID > 263 OR DOLocationID > 263").count())

Ham: 38310226
Negatif ücret: 382882
Yıl != 2023: 104


Py4JJavaError: An error occurred while calling o922.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 11 in stage 66.0 failed 1 times, most recent failure: Lost task 11.0 in stage 66.0 (TID 485) (10.2.146.135 executor driver): org.apache.spark.SparkException: Parquet column cannot be converted in file file:///c:/Users/iclal/Desktop/NYC-Taxi-Bench/data/raw/yellow_tripdata_2023-12.parquet. Column: [PULocationID], Expected: bigint, Found: INT32.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.unsupportedSchemaColumnConvertError(QueryExecutionErrors.scala:854)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.nextIterator(FileScanRDD.scala:287)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:129)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1

DataFusion


In [8]:
# =====================================================
# DataFusion Smoke Test - Bölüm 3.3.3 için demo run
# =====================================================
from datafusion import SessionContext
import time

ctx = SessionContext()

ctx.register_parquet(
    "trips",
    "data/clean/yellow_tripdata_2023/pickup_year=2023/pickup_month=1"
)

# ÖNEMLİ: DataFusion ANSI SQL uyumludur, karışık case kolon adları çift tırnak ister
query = """
    SELECT "PULocationID", "DOLocationID",
           ROUND(CAST(AVG(total_amount) AS DECIMAL(10,2)), 2) AS avg_fare_usd,
           ROUND(AVG(trip_duration_seconds)/60.0, 2) AS avg_minutes,
           COUNT(*) AS trip_count
    FROM trips
    GROUP BY "PULocationID", "DOLocationID"
    ORDER BY trip_count DESC
    LIMIT 5
"""

t0 = time.perf_counter()
result_df = ctx.sql(query).to_pandas()
elapsed = time.perf_counter() - t0

print(f"DataFusion sorgu süresi: {elapsed:.3f} saniye")
print(f"Satır sayısı: {len(result_df)}")
print("\nSonuçlar:")
print(result_df)

DataFusion sorgu süresi: 0.246 saniye
Satır sayısı: 5

Sonuçlar:
   PULocationID  DOLocationID avg_fare_usd  avg_minutes  trip_count
0           237           236        15.43         6.98       44162
1           236           237        15.99         8.04       37646
2           236           236        13.21         4.95       28454
3           237           237        13.64         5.55       27702
4           237           161        16.22         8.75       18604


Spark

In [9]:
# =====================================================
# Spark Karşılaştırma Testi - aynı sorgu Spark'ta
# =====================================================
from pyspark.sql import functions as F
import time

# Aynı dosyayı Spark ile oku ve aynı sorguyu çalıştır
t0 = time.perf_counter()

sdf = spark.read.parquet("data/clean/yellow_tripdata_2023/pickup_year=2023/pickup_month=1")

result_spark = (sdf
    .groupBy("PULocationID", "DOLocationID")
    .agg(
        F.round(F.avg("total_amount"), 2).alias("avg_fare_usd"),
        F.round(F.avg("trip_duration_seconds") / 60.0, 2).alias("avg_minutes"),
        F.count("*").alias("trip_count")
    )
    .orderBy(F.col("trip_count").desc())
    .limit(5)
    .toPandas())

elapsed_spark = time.perf_counter() - t0

print(f"Spark sorgu süresi (soğuk cache): {elapsed_spark:.3f} saniye")
print("\nSonuçlar:")
print(result_spark)

Spark sorgu süresi (soğuk cache): 1.841 saniye

Sonuçlar:
   PULocationID  DOLocationID  avg_fare_usd  avg_minutes  trip_count
0           237           236         15.43         6.98       44162
1           236           237         15.99         8.04       37646
2           236           236         13.21         4.95       28454
3           237           237         13.64         5.55       27702
4           237           161         16.22         8.75       18604
